# Paper 3 — 03 · RD-DPO training

Trains one (anchor, condition, seed) tuple per run. Conditions:

| Condition           | Layers updated                              | Description                  |
|---------------------|---------------------------------------------|------------------------------|
| `sft`               | All linear modules, all blocks (LoRA r=16)  | SFT on chosen only           |
| `rd-dpo-k1/2/4/8`   | LoRA r=16 on top-k probe-selected blocks    | RD-DPO with k targeted blocks|
| `dpo-lora-all`      | All linear modules, all blocks (LoRA r=16)  | Standard LoRA-DPO baseline   |
| `dpo-full`          | Every parameter                             | Vanilla full-model DPO       |

**A100-40G runtime** (one epoch, ~12k pairs, seq 1024, effective batch 32):

| Anchor       | dpo-full | dpo-lora-all | rd-dpo-k4 |
|--------------|----------|--------------|-----------|
| Qwen-2.5-3B  | ~3 h     | ~2 h         | ~1 h      |
| Llama-3.2-3B | ~3 h     | ~2 h         | ~1 h      |
| Gemma-3-4B   | ~4 h     | ~2.5 h       | ~1.3 h    |

In [1]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    'torchao>=0.16' \
    huggingface_hub ipywidgets pyyaml -q


In [2]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}


# --- Paper 3 single-source-of-truth helpers ---
# short_of and family_of imported from src/prompts.py so the notebook
# convention stays aligned with notebook 02 and notebook 02b.
PROMPTS_SRC_DIR = DRIVE_ROOT / "src"
if not (PROMPTS_SRC_DIR / "prompts.py").exists():
    raise RuntimeError(
        f"src/prompts.py not found at {PROMPTS_SRC_DIR / 'prompts.py'}.\n"
        "Copy it from your local rosafety-align/src/prompts.py to that "
        "path in Drive, then re-run this cell."
    )
sys.path.insert(0, str(PROMPTS_SRC_DIR))
from prompts import (
    short_of, family_of,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    verify_smoke_gate, SmokeGateNotPassed,
)


# Colab clears the volatile /tmp hf_datasets cache between write and read
# during DPOTrainer precompute_ref_log_probs, raising FileNotFoundError on
# cache-*.arrow. Disable on-disk datasets caching for the whole session so
# the small in-memory preference datasets stay in RAM.
from datasets import disable_caching
disable_caching()

print("Bootstrap done.")


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.11.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.


## Pre-flight gate

Verifies smoke gate is fresh, the augmented preferences file
(`preferences_v2.jsonl`) exists, and the probe artefact
(`selected_blocks.json`) exists for the configured anchor.
Training will not start without all three.


In [3]:
ANCHOR_PREFLIGHT = "Qwen/Qwen2.5-3B-Instruct"   # match cfg cell
short_pre  = short_of(ANCHOR_PREFLIGHT)

print("Pre-flight gate (training; PAPER3_PLAN section 15.7)")
print("-" * 72)

# Training reads FROZEN preference data and makes no OpenRouter teacher
# calls, so the smoke-gate freshness window (which guards nb02/02b
# generation spend) is ADVISORY here, not fatal. We still surface its
# status so prompt-digest drift stays visible.
try:
    smoke = verify_smoke_gate(PREFS_DIR, ANCHOR_PREFLIGHT)
    print(f"smoke gate             : PASS ({smoke['completed_at']})")
except SmokeGateNotPassed as e:
    smoke = None
    print(f"smoke gate             : SKIPPED (non-fatal for training) - {e}")

# Load-bearing for training: preference data (e6 or x4) and the probe.
prefs_dir_pre = PREFS_DIR / short_pre
prefs_present = sorted(
    p.name for p in [prefs_dir_pre / "preferences_v2.jsonl",
                     prefs_dir_pre / "preferences_x4.jsonl"] if p.exists()
)
if not prefs_present:
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"No preference data in {prefs_dir_pre} (need preferences_v2.jsonl "
        f"for e6 or preferences_x4.jsonl for x4 / rank sweeps).\n"
        f"Run notebook 02b (Stage 4 augmentation) for {short_pre} first."
    )

probe_path_pre = PROBE_DIR / short_pre / "selected_blocks.json"
if not probe_path_pre.exists():
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"Probe artefacts not found at {probe_path_pre}.\n"
        f"Run notebook 01 (refusal-direction probe) for {short_pre} first."
    )

print(f"preferences found      : {prefs_present}")
print(f"selected_blocks found  : {probe_path_pre}")
print(f"\nPRE-FLIGHT PASS. Anchor {short_pre!r} is ready for training.")

Pre-flight gate (training; PAPER3_PLAN section 15.7)
------------------------------------------------------------------------
smoke gate             : SKIPPED (non-fatal for training) - Smoke file at /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/qwen2.5-3b/smoke.json is 59 days old (threshold 7 days). Re-run notebook 00.
preferences found      : ['preferences_v2.jsonl', 'preferences_x4.jsonl']
selected_blocks found  : /content/drive/MyDrive/PhD/paper3-alignment/data/probes/qwen2.5-3b/selected_blocks.json

PRE-FLIGHT PASS. Anchor 'qwen2.5-3b' is ready for training.


## Configuration

`SEED` cycles `{17, 1729, 65537}` per `configs/models.yaml`.

In [4]:
ANCHOR    = "Qwen/Qwen2.5-3B-Instruct"
CONDITION = "rd-dpo-k4-bal-b03"
SEED      = 17

# Locked at week 1 (METHOD §4.2). Per-anchor tuning of these is forbidden.
BETA              = 0.3       # bumped from 0.1 on 2026-05-13. The 0.1
                             # rebalanced run regressed jailbreak by 10pp
                             # and left toxicity 3.8pp below base; a higher
                             # beta keeps the policy closer to the base
                             # reference, muting drift caused by an
                             # imperfect data mix.
LR                = 5e-6
EPOCHS            = 3        # small dataset -> need more epochs
WARMUP_STEPS      = 10       # was 100; never fired at 5-6 steps/epoch
MAX_STEPS         = -1       # -1 = no cap (HF convention); EPOCHS rules.
                             # The first live run with MAX_STEPS=200 ran
                             # ~36 epochs on the 178-pair train set and
                             # over-fit (accuracy hit 1.0 by step 100).
                             # logging_steps=2 below: the second run only
                             # took 18 steps; with logging_steps=25 the
                             # trajectory was empty.
MAX_SEQ_LEN       = 1024
BATCH_PER_DEV     = 4
GRAD_ACCUM_STEPS  = 8       # effective batch 32
LORA_R, LORA_A    = 16, 32
LORA_DROPOUT      = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

RUN_TAG = f"{short}__{CONDITION}__seed{SEED}"
RUN_DIR = ADAPTERS_DIR / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"run    : {RUN_TAG}")
print(f"adapter: {RUN_DIR}")


run    : qwen2.5-3b__rd-dpo-k4-bal-b03__seed17
adapter: /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17


## Resolve probe-selected blocks

For RD-DPO conditions only — `dpo-lora-all` / `dpo-full` / `sft` ignore.

In [5]:
selected_blocks = None
if CONDITION.startswith("rd-dpo-"):
    # CONDITION is e.g. 'rd-dpo-k4' or 'rd-dpo-k4-bal'. Strip the
    # 'rd-dpo-k' prefix and take the leading integer; anything after
    # the first non-digit (like '-bal') is a tag, not part of k.
    import re
    m = re.match(r"rd-dpo-k(\d+)", CONDITION)
    if not m:
        raise ValueError(f"could not parse k from CONDITION={CONDITION!r}")
    k = int(m.group(1))
    sel_path = PROBE_DIR / short / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(
            f"Probe artefact not found at {sel_path}. "
            f"Run 01_refusal_probe.ipynb first."
        )
    sel = json.loads(sel_path.read_text())
    selected_blocks = sel[str(k)]
    print(f"selected blocks (k={k}): {selected_blocks}")
else:
    print("(non-RD condition; LoRA targets all blocks or full model)")

selected blocks (k=4): [31, 32, 33, 34]


## Load preferences + build dataset

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

pairs_path = PREFS_DIR / short / "preferences_v2.jsonl"
if not pairs_path.exists():
    raise FileNotFoundError(
        f"Preference dataset not found at {pairs_path}. "
        f"Run notebook 02 (Stage 1+2) and notebook 02b (Stage 4 augmentation) first."
    )
pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
print(f"loaded {len(pairs)} pairs from {pairs_path}")

# --- 50/50 rebalance between refuse-side and help-side streams ---
# 2026-05-13 finding: at 53/47 help-vs-refuse on Qwen the recipe regressed
# toxicity refusal by 11.3 pp on the holdout split (results/...delta_vs_base.json).
# Hypothesis: the 99 over-refusal counter pairs out-pulled the 88 refuse-side
# pairs and the model generalised "do not produce RO apologetic-refusal"
# across the toxicity holdout. Down-sample the larger side to match the
# smaller side. Same probe, same beta, same epochs, same seed.
import random
REFUSE_SOURCES = {"core_s2", "xl"}
HELP_SOURCES   = {"overref"}
refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
if other:
    raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
n_target = min(len(refuse), len(helpp))
rng = random.Random(SEED)
refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
pairs = refuse_kept + helpp_kept
rng.shuffle(pairs)
from collections import Counter
src_counts = Counter(r["meta"]["source"] for r in pairs)
print(f"rebalanced to {len(pairs)} pairs ({n_target} refuse-side + {n_target} help-side)")
print(f"  source counts: {dict(src_counts)}")

tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token

def format_row(r):
    prompt_chat = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

ds_full   = Dataset.from_list([format_row(r) for r in pairs])
splits    = ds_full.train_test_split(test_size=0.05, seed=SEED)
ds_train  = splits["train"]; ds_eval = splits["test"]
print({"train": len(ds_train), "eval": len(ds_eval)})


loaded 187 pairs from /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/qwen2.5-3b/preferences_v2.jsonl
rebalanced to 176 pairs (88 refuse-side + 88 help-side)
  source counts: {'core_s2': 65, 'overref': 88, 'xl': 23}


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{'train': 167, 'eval': 9}


## Build LoRA target spec

PEFT's `target_modules` accepts a list of module name suffixes or a regex. For
RD-DPO we restrict by **block index** via a regex matching `layers.{i}.…`.

In [7]:
from peft import LoraConfig

if CONDITION == "dpo-full":
    lora_config = None
elif CONDITION in ("sft", "dpo-lora-all"):
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    )
elif CONDITION.startswith("rd-dpo-"):
    block_pat = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(LORA_TARGET_MODULES)
    target_re = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )
else:
    raise ValueError(f"unknown condition {CONDITION!r}")
print("lora_config:", lora_config)

lora_config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='^.*\\.layers\\.(31|32|33|34)\\.(?:.*\\.)?(?:q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## Load model + train

A100 footprint:
- bf16 throughout (Paper 2 R-Gemma-3 lesson: never fp16).
- Gradient checkpointing on.
- TRL precomputes reference log-probs on dev (avoids carrying the reference model forward through every step).
- For `dpo-full`, expect ~38 GB peak; for LoRA conditions, ~16-22 GB.

In [8]:
from transformers import AutoModelForCausalLM, set_seed
from trl import DPOConfig, DPOTrainer

set_seed(SEED)

model = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

if CONDITION == "sft":
    from trl import SFTTrainer, SFTConfig
    sft_cfg = SFTConfig(
        output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_PER_DEV,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=SEED, report_to=["none"],
        dataset_text_field="chosen", max_length=MAX_SEQ_LEN,
        max_steps=MAX_STEPS,
    )
    trainer = SFTTrainer(
        model=model, args=sft_cfg, peft_config=lora_config,
        train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok,
    )
else:
    dpo_cfg = DPOConfig(
        output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_PER_DEV,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=SEED, report_to=["none"],
        beta=BETA, loss_type="sigmoid",
        max_length=MAX_SEQ_LEN,
        precompute_ref_log_probs=False,
        max_steps=MAX_STEPS,
    )
    trainer = DPOTrainer(
        model=model, args=dpo_cfg, peft_config=lora_config,
        train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok,
    )

print("starting training...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"training done in {elapsed/60:.1f} min")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Tokenizing eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
2,0.693147
4,0.705333
6,0.723402
8,0.714039
10,0.703300
12,0.732962
14,0.685201
16,0.703933
18,0.688832


training done in 2.1 min


## Save adapter + run-meta

In [9]:
trainer.save_model(str(RUN_DIR))
peak_mem = torch.cuda.max_memory_allocated() / 1e9

(RUN_DIR / "run_meta.json").write_text(json.dumps({
    "run_tag": RUN_TAG, "anchor": ANCHOR, "condition": CONDITION, "seed": SEED,
    "selected_blocks": selected_blocks,
    "preference_dataset": str(pairs_path),
    "n_pairs": len(pairs),
    "pair_source_counts": dict(src_counts),
    "training_compute": {
        "wallclock_seconds": round(elapsed, 1),
        "peak_gpu_memory_gb": round(peak_mem, 2),
        "device": DEVICE_NAME,
    },
    "hyperparams": {
        "beta": BETA, "lr": LR, "epochs": EPOCHS,
        "lora_r": LORA_R, "lora_alpha": LORA_A,
        "max_seq_len": MAX_SEQ_LEN,
        "per_device_train_batch_size": BATCH_PER_DEV,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    },
    "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, indent=2))
print(f"saved adapter + meta -> {RUN_DIR}")

/tmp/ipykernel_1952/67487776.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


saved adapter + meta -> /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17


## Batch run: Llama + Gemma at the rebalanced beta=0.1 recipe

Independent of the Qwen-only cells above. Trains the two remaining
anchors back-to-back so the operator can leave Colab unattended for
~10 minutes total. Same probe per anchor (loaded from
`data/probes/<short>/selected_blocks.json`), same hyperparameters
as the rebalanced Qwen run that produced the least-bad delta-vs-base.

Each anchor saves to its own adapter folder; the cell is a no-op
for an anchor that already has its run_meta.json on disk, so safe
to re-execute.


In [10]:
import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# ----- locked recipe (mirrors the Qwen rebalanced run) -----
# --- defensive: ensure prompts.py is importable when run standalone ---
# (no-op if bootstrap already ran in this kernel)
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

from prompts import ANCHORS as BATCH_ANCHORS
BATCH_CONDITION = "rd-dpo-k4-bal-e6-mid"
BATCH_SEED      = 17

BATCH_BETA              = 0.1
BATCH_LR                = 5e-6
BATCH_EPOCHS            = 6
BATCH_WARMUP_STEPS      = 2
BATCH_MAX_STEPS         = -1
BATCH_MAX_SEQ_LEN       = 1024
BATCH_BATCH_PER_DEV     = 4
BATCH_GRAD_ACCUM_STEPS  = 8
BATCH_LORA_R            = 16
BATCH_LORA_A            = 32
BATCH_LORA_DROPOUT      = 0.05
BATCH_LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "xl"}
HELP_SOURCES   = {"overref"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    run_tag  = f"{short_a}__{BATCH_CONDITION}__seed{BATCH_SEED}"
    run_dir  = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor}] run_meta.json already present at {run_dir}; skipping (delete to force rerun).")
        return

    print(f"\n=== training {anchor} -> {run_tag} ===")

    # Resolve probe-selected blocks
    m = re.match(r"rd-dpo-k(\d+)", BATCH_CONDITION)
    if not m:
        raise ValueError(f"could not parse k from {BATCH_CONDITION!r}")
    k = int(m.group(1))
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if BATCH_CONDITION.endswith("-mid") or "-mid-" in BATCH_CONDITION:
        sel_path = PROBE_DIR / short_a / "selected_blocks_mid.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"probe artefact not found at {sel_path}; run notebook 01 first")
    selected_blocks = json.loads(sel_path.read_text())[str(k)]
    print(f"  probe blocks (k={k}): {selected_blocks}")

    # Load + rebalance preferences
    pairs_path = PREFS_DIR / short_a / "preferences_v2.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run notebook 02 + 02b first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, BATCH_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target}); sources={dict(src_counts)}")

    # Tokenize + chat-template
    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=BATCH_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    # LoRA target spec
    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(BATCH_LORA_TARGET_MODULES)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=BATCH_LORA_R, lora_alpha=BATCH_LORA_A, lora_dropout=BATCH_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(BATCH_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=BATCH_EPOCHS,
        per_device_train_batch_size=BATCH_BATCH_PER_DEV,
        gradient_accumulation_steps=BATCH_GRAD_ACCUM_STEPS,
        learning_rate=BATCH_LR, lr_scheduler_type="cosine",
        warmup_steps=BATCH_WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=BATCH_SEED, report_to=["none"],
        beta=BATCH_BETA, loss_type="sigmoid",
        max_length=BATCH_MAX_SEQ_LEN,
        precompute_ref_log_probs=False,
        max_steps=BATCH_MAX_STEPS,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": BATCH_CONDITION, "seed": BATCH_SEED,
        "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": BATCH_BETA, "lr": BATCH_LR, "epochs": BATCH_EPOCHS,
            "lora_r": BATCH_LORA_R, "lora_alpha": BATCH_LORA_A,
            "max_seq_len": BATCH_MAX_SEQ_LEN,
            "per_device_train_batch_size": BATCH_BATCH_PER_DEV,
            "gradient_accumulation_steps": BATCH_GRAD_ACCUM_STEPS,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    # Free GPU before next anchor
    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in BATCH_ANCHORS:
    _train_one(anchor)
print("\nbatch training done.")


[Qwen/Qwen2.5-3B-Instruct] run_meta.json already present at /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-e6-mid__seed17; skipping (delete to force rerun).
[meta-llama/Llama-3.2-3B-Instruct] run_meta.json already present at /content/drive/MyDrive/PhD/paper3-alignment/adapters/llama-3.2-3b__rd-dpo-k4-bal-e6-mid__seed17; skipping (delete to force rerun).
[google/gemma-3-4b-it] run_meta.json already present at /content/drive/MyDrive/PhD/paper3-alignment/adapters/gemma-3-4b__rd-dpo-k4-bal-e6-mid__seed17; skipping (delete to force rerun).

batch training done.


## LR sweep: Qwen + Llama at LR in {1e-5, 2e-5, 5e-5}

Tests whether Qwen and Llama can be trained on the same
rebalanced 176-200 preference pairs by raising the LR. The
default recipe (LR=5e-6) left both at the random baseline
while Gemma converged. If at LR=1e-5 or 2e-5 either anchor
trains to margins > 0.1, the recipe is fine and the gap
between Gemma and the others was just LR.

Six adapters total. Saves to
`adapters/<short>__rd-dpo-k4-bal-e6-lr{tag}__seed17/`.
Skips an anchor that already has its run_meta.json.

Probe selection: top-of-net (`selected_blocks.json`). The
mid-depth ablation rejected the depth hypothesis, so we go
back to the high-score top-of-net selection.


In [11]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# Sweep configuration: only Qwen + Llama (Gemma already trained, no point retesting)
SWEEP_ANCHORS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]
SWEEP_LRS = [
    (1e-5, "1e5"),
    (2e-5, "2e5"),
    (5e-5, "5e5"),
]

SWEEP_SEED  = 17
SWEEP_BETA  = 0.1
SWEEP_EPOCHS  = 6
SWEEP_WARMUP  = 2
SWEEP_MAX_SEQ = 1024
SWEEP_BATCH   = 4
SWEEP_GA      = 8
SWEEP_LORA_R  = 16
SWEEP_LORA_A  = 32
SWEEP_LORA_DROPOUT = 0.05
SWEEP_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "xl"}
HELP_SOURCES   = {"overref"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor, lr, lr_tag):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    condition = f"rd-dpo-k4-bal-e6-lr{lr_tag}"
    run_tag   = f"{short_a}__{condition}__seed{SWEEP_SEED}"
    run_dir   = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor} @ lr={lr:.0e}] {run_dir.name} already exists; skipping.")
        return

    print(f"\n=== training {anchor} @ lr={lr:.0e} -> {run_tag} ===")

    # Probe-selected blocks (top-of-net, original selection)
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"probe artefact not found at {sel_path}; run nb01 first")
    selected_blocks = json.loads(sel_path.read_text())["4"]
    print(f"  probe blocks: {selected_blocks}")

    # Load + rebalance preferences
    pairs_path = PREFS_DIR / short_a / "preferences_v2.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run nb02 + nb02b first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, SWEEP_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target}); sources={dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=SWEEP_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(SWEEP_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=SWEEP_LORA_R, lora_alpha=SWEEP_LORA_A, lora_dropout=SWEEP_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(SWEEP_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=SWEEP_EPOCHS,
        per_device_train_batch_size=SWEEP_BATCH,
        gradient_accumulation_steps=SWEEP_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=SWEEP_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=SWEEP_SEED, report_to=["none"],
        beta=SWEEP_BETA, loss_type="sigmoid",
        max_length=SWEEP_MAX_SEQ,
        precompute_ref_log_probs=False,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": condition,
        "seed": SWEEP_SEED, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": SWEEP_BETA, "lr": lr, "epochs": SWEEP_EPOCHS,
            "warmup_steps": SWEEP_WARMUP,
            "lora_r": SWEEP_LORA_R, "lora_alpha": SWEEP_LORA_A,
            "max_seq_len": SWEEP_MAX_SEQ,
            "per_device_train_batch_size": SWEEP_BATCH,
            "gradient_accumulation_steps": SWEEP_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in SWEEP_ANCHORS:
    for lr, lr_tag in SWEEP_LRS:
        _train_one(anchor, lr, lr_tag)
print("\nLR sweep training done.")


[Qwen/Qwen2.5-3B-Instruct @ lr=1e-05] qwen2.5-3b__rd-dpo-k4-bal-e6-lr1e5__seed17 already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct @ lr=2e-05] qwen2.5-3b__rd-dpo-k4-bal-e6-lr2e5__seed17 already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct @ lr=5e-05] qwen2.5-3b__rd-dpo-k4-bal-e6-lr5e5__seed17 already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct @ lr=1e-05] llama-3.2-3b__rd-dpo-k4-bal-e6-lr1e5__seed17 already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct @ lr=2e-05] llama-3.2-3b__rd-dpo-k4-bal-e6-lr2e5__seed17 already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct @ lr=5e-05] llama-3.2-3b__rd-dpo-k4-bal-e6-lr5e5__seed17 already exists; skipping.

LR sweep training done.


## x4 expansion: train Qwen / Llama / Gemma on preferences_x4

Loops over `ANCHORS` and trains each on `preferences_x4.jsonl`
at the LR that worked for it (Gemma 5e-6, Qwen+Llama 2e-5).
Same probe (top-of-net), same beta=0.1, same epochs=6, same
rebalance step, same seed=17.

Adapter folder: `adapters/<short>__rd-dpo-k4-bal-e6-x4__seed17/`.
Skips an anchor whose `run_meta.json` already exists.


In [12]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

from prompts import ANCHORS

# Per-anchor working LR, established by the LR sweep (rd-dpo-k4-bal-e6-lr*).
PER_ANCHOR_LR = {
    "Qwen/Qwen2.5-3B-Instruct":          2e-5,
    "meta-llama/Llama-3.2-3B-Instruct":  2e-5,
    "google/gemma-3-4b-it":               5e-6,
}

X4_CONDITION = "rd-dpo-k4-bal-e6-x4"
X4_SEED = 17
X4_BETA = 0.1
X4_EPOCHS = 6
X4_WARMUP = 2
X4_BATCH = 4
X4_GA = 8
X4_MAX_SEQ = 1024
X4_LORA_R = 16
X4_LORA_A = 32
X4_LORA_DROPOUT = 0.05
X4_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref", "overref_ext"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    lr = PER_ANCHOR_LR[anchor]
    run_tag = f"{short_a}__{X4_CONDITION}__seed{X4_SEED}"
    run_dir = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor}] run_meta.json already exists; skipping.")
        return

    print(f"\n=== x4 training {anchor} @ lr={lr:.0e} -> {run_tag} ===")

    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"probe artefact not found at {sel_path}")
    selected_blocks = json.loads(sel_path.read_text())["4"]

    pairs_path = PREFS_DIR / short_a / "preferences_x4.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run x4 batch in nb02 first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, X4_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")
    print(f"  source distribution: {dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=X4_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(X4_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=X4_LORA_R, lora_alpha=X4_LORA_A, lora_dropout=X4_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(X4_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=X4_EPOCHS,
        per_device_train_batch_size=X4_BATCH,
        gradient_accumulation_steps=X4_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=X4_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=X4_SEED, report_to=["none"],
        beta=X4_BETA, loss_type="sigmoid",
        max_length=X4_MAX_SEQ,
        precompute_ref_log_probs=False,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": X4_CONDITION,
        "seed": X4_SEED, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": X4_BETA, "lr": lr, "epochs": X4_EPOCHS,
            "warmup_steps": X4_WARMUP,
            "lora_r": X4_LORA_R, "lora_alpha": X4_LORA_A,
            "max_seq_len": X4_MAX_SEQ,
            "per_device_train_batch_size": X4_BATCH,
            "gradient_accumulation_steps": X4_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in ANCHORS:
    _train_one(anchor)
print("\nx4 batch training done.")


[Qwen/Qwen2.5-3B-Instruct] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct] run_meta.json already exists; skipping.
[google/gemma-3-4b-it] run_meta.json already exists; skipping.

x4 batch training done.


## Rank sweep: train Qwen / Llama / Gemma at LoRA r=64 and r=128 on x4 data

Tests whether the LoRA r=16 trainable subspace is the bottleneck for
held-out cross-lingual refusal. Six new adapters (3 anchors x 2 ranks)
on the same x4 expanded preference data, same best-LR per anchor, same
top-of-network probe layers as the r=16 x4 baseline. The cell is
idempotent: skips runs whose `run_meta.json` already exists.

Total cost: ~5 A100-hours. Outputs land in
`adapters/{short}__rd-dpo-k4-bal-e6-x4-r{rank}__seed17/`.

In [13]:
"""
Rank-sweep batch cell.

Trains six new adapters (3 anchors x 2 ranks: r=64, r=128) at the
x4 expanded preference data and best-LR per anchor.

Reuses everything from the x4 cell:
  - PER_ANCHOR_LR
  - hyperparameters (seed, beta, epochs, warmup, batch, ga, max_seq)
  - REFUSE_SOURCES, HELP_SOURCES, _rebalance()

Adapter naming: rd-dpo-k4-bal-e6-x4-r{rank}__seed17

Estimated cost: ~5 A100-hours total.
"""

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# ANCHORS, ADAPTERS_DIR, PREFS_DIR, short_of, family_of, load_kwargs_for,
# DEVICE_NAME, PROBE_DIR are defined in earlier cells.
from prompts import ANCHORS

# --- rank sweep config ---
RANK_SWEEP   = [64, 128]   # add to the existing r=16 baseline
RS_BASE_COND = "rd-dpo-k4-bal-e6-x4"   # source data condition
PER_ANCHOR_LR = {
    "Qwen/Qwen2.5-3B-Instruct":          2e-5,
    "meta-llama/Llama-3.2-3B-Instruct":  2e-5,
    "google/gemma-3-4b-it":               5e-6,
}

# Inherit hyperparameters from x4 cell
RS_SEED = 17
RS_BETA = 0.1
RS_EPOCHS = 6
RS_WARMUP = 2
RS_BATCH = 4
RS_GA = 8
RS_MAX_SEQ = 1024
RS_LORA_DROPOUT = 0.05
RS_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref", "overref_ext"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one_rank(anchor, lora_r):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    lr = PER_ANCHOR_LR[anchor]
    cond_tag = f"{RS_BASE_COND}-r{lora_r}"
    run_tag = f"{short_a}__{cond_tag}__seed{RS_SEED}"
    run_dir = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor} r={lora_r}] run_meta.json already exists; skipping.")
        return

    # Probe layers (same as x4 / e6: top-of-net selection per anchor)
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"{sel_path} missing; run nb01 probe first")
    selected_blocks = json.loads(sel_path.read_text())["4"]

    # Load x4 preferences (the same data as the r=16 x4 baseline)
    pairs_path = PREFS_DIR / short_a / "preferences_x4.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(
            f"{pairs_path} missing; run the x4 batch in nb02 (and the x4 train "
            f"cell in this notebook) before the rank sweep."
        )
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, RS_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"=== rank-sweep training {anchor} @ r={lora_r}, lr={lr} ->")
    print(f"    {run_tag}")
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")
    print(f"  source distribution: {dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=RS_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(RS_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    # Standard LoRA scaling: alpha = 2 * rank keeps effective magnitude
    # comparable across ranks (matches r=16, alpha=32 baseline).
    lora_alpha = 2 * lora_r
    lora_cfg = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, lora_dropout=RS_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(RS_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=RS_EPOCHS,
        per_device_train_batch_size=RS_BATCH,
        gradient_accumulation_steps=RS_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=RS_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=RS_SEED, report_to=["none"],
        beta=RS_BETA, loss_type="sigmoid",
        max_length=RS_MAX_SEQ,
        precompute_ref_log_probs=False,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": cond_tag,
        "seed": RS_SEED, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": RS_BETA, "lr": lr, "epochs": RS_EPOCHS,
            "warmup_steps": RS_WARMUP,
            "lora_r": lora_r, "lora_alpha": lora_alpha,
            "max_seq_len": RS_MAX_SEQ,
            "per_device_train_batch_size": RS_BATCH,
            "gradient_accumulation_steps": RS_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


# Loop: 3 anchors × 2 new ranks = 6 adapters
# Order: smaller rank first per anchor, so all anchors get r=64 before any r=128
# (lets you stop early if r=64 already shows the answer).
for lora_r in RANK_SWEEP:
    for anchor in ANCHORS:
        _train_one_rank(anchor, lora_r)
print("\nrank-sweep batch training done.")

[Qwen/Qwen2.5-3B-Instruct r=64] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct r=64] run_meta.json already exists; skipping.
[google/gemma-3-4b-it r=64] run_meta.json already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct r=128] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct r=128] run_meta.json already exists; skipping.
[google/gemma-3-4b-it r=128] run_meta.json already exists; skipping.

rank-sweep batch training done.


## Seed sweep: train each load-bearing condition at seed {1729, 65537}

Adds 12 new training runs to convert the load-bearing dissociation cells
(3 anchors x {e6, x4}) from single-seed to three-seed (17 + 1729 + 65537,
the seeds pre-registered in `configs/models.yaml`). Hyperparameters are
identical to the seed=17 baselines listed in `tables/multi-anchor-delta`
and `tables/x4-comparison`; only the seed varies.

The cell is idempotent: each (anchor, scale, seed) triple is skipped if
its `run_meta.json` already exists. Estimated cost: ~7-9 A100-hours.

Source mirror at `experiments/_seed_sweep_train_cell.py`.


In [14]:
"""
Seed-sweep training cell.

Trains 12 new adapters: 3 anchors x 2 data conditions (e6, x4) x
2 new seeds (1729, 65537), at the best-LR per anchor and otherwise
identical hyperparameters to the seed=17 baselines that appear in
the multi-anchor delta and x4-comparison manuscript tables.

The point: convert the load-bearing dissociation cells from
single-seed to three-seed (17 + 1729 + 65537), so the cross-anchor
cross-lingual non-lift becomes a 9-cell sign test instead of a
3-cell one.

Reused symbols (defined in earlier cells of nb03):
  - ANCHORS (from prompts)
  - short_of, family_of, load_kwargs_for, DEVICE_NAME
  - ADAPTERS_DIR, PREFS_DIR, PROBE_DIR

Adapter naming:
  e6 conditions (mirror existing best-LR-seed17 tags):
    qwen2.5-3b__rd-dpo-k4-bal-e6-lr2e5__seed{1729,65537}
    llama-3.2-3b__rd-dpo-k4-bal-e6-lr2e5__seed{1729,65537}
    gemma-3-4b__rd-dpo-k4-bal-e6__seed{1729,65537}
  x4 conditions:
    {short}__rd-dpo-k4-bal-e6-x4__seed{1729,65537}

Estimated cost: ~7-9 A100-hours total.
"""

# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

from prompts import ANCHORS

# --- seed sweep config ---
# Seeds {17, 1729, 65537} are pre-registered in configs/models.yaml.
# Seed 17 already exists; we add the other two.
NEW_SEEDS = [1729, 65537]

# Per-anchor best-LR + e6 condition tag (mirrors the seed=17 baseline
# adapters cited in tables/multi-anchor-delta and tables/x4-comparison).
PER_ANCHOR_E6 = {
    # anchor                              : (condition_tag,                   lr)
    "Qwen/Qwen2.5-3B-Instruct":            ("rd-dpo-k4-bal-e6-lr2e5",       2e-5),
    "meta-llama/Llama-3.2-3B-Instruct":    ("rd-dpo-k4-bal-e6-lr2e5",       2e-5),
    "google/gemma-3-4b-it":                ("rd-dpo-k4-bal-e6",             5e-6),
}
# x4 condition: shared tag across anchors; LR per-anchor matches PER_ANCHOR_E6.
PER_ANCHOR_X4 = {
    "Qwen/Qwen2.5-3B-Instruct":            ("rd-dpo-k4-bal-e6-x4",          2e-5),
    "meta-llama/Llama-3.2-3B-Instruct":    ("rd-dpo-k4-bal-e6-x4",          2e-5),
    "google/gemma-3-4b-it":                ("rd-dpo-k4-bal-e6-x4",          5e-6),
}

# preferences file per data scale
PREFS_FILE = {
    "e6": "preferences_v2.jsonl",
    "x4": "preferences_x4.jsonl",
}

# Hyperparameters: identical to the seed=17 baselines for these condition tags.
SS_BETA            = 0.1
SS_EPOCHS          = 6
SS_WARMUP          = 2
SS_BATCH           = 4
SS_GA              = 8
SS_MAX_SEQ         = 1024
SS_LORA_R          = 16
SS_LORA_A          = 32
SS_LORA_DROPOUT    = 0.05
SS_LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref", "overref_ext"}


def _rebalance(pairs, seed):
    """Same rebalance routine as the seed=17 baselines: down-sample
    the larger of refuse-side / help-side to match. Seed-keyed RNG
    so different seeds see slightly different rebalance subsamples,
    which is part of the variance we are trying to characterise.
    """
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor, data_scale, seed):
    """Train one (anchor, data_scale, seed) triple. Idempotent: skips
    runs whose run_meta.json already exists."""
    short_a   = short_of(anchor)
    family_a  = family_of(anchor)
    cond_map  = PER_ANCHOR_E6 if data_scale == "e6" else PER_ANCHOR_X4
    cond_tag, lr = cond_map[anchor]
    run_tag   = f"{short_a}__{cond_tag}__seed{seed}"
    run_dir   = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor} {data_scale} seed{seed}] run_meta.json already exists; skipping.")
        return

    print(f"\n=== seed-sweep training {anchor} @ {data_scale} seed={seed} lr={lr:.0e}")
    print(f"    -> {run_tag}")

    # Probe layers (same selection as seed=17 baseline: top-of-net)
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"{sel_path} missing; run nb01 probe first")
    selected_blocks = json.loads(sel_path.read_text())["4"]

    # Load preferences for the data scale
    pairs_path = PREFS_DIR / short_a / PREFS_FILE[data_scale]
    if not pairs_path.exists():
        raise FileNotFoundError(
            f"{pairs_path} missing; run nb02 (and nb02b for x4) first"
        )
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, seed)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")
    print(f"  source distribution: {dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=seed)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(SS_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=SS_LORA_R, lora_alpha=SS_LORA_A, lora_dropout=SS_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(seed)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=SS_EPOCHS,
        per_device_train_batch_size=SS_BATCH,
        gradient_accumulation_steps=SS_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=SS_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=seed, report_to=["none"],
        beta=SS_BETA, loss_type="sigmoid",
        max_length=SS_MAX_SEQ,
        precompute_ref_log_probs=False,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": cond_tag,
        "seed": seed, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": SS_BETA, "lr": lr, "epochs": SS_EPOCHS,
            "warmup_steps": SS_WARMUP,
            "lora_r": SS_LORA_R, "lora_alpha": SS_LORA_A,
            "max_seq_len": SS_MAX_SEQ,
            "per_device_train_batch_size": SS_BATCH,
            "gradient_accumulation_steps": SS_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


# Loop order: e6 first across all anchors (cheaper, smoke-tests pipeline),
# then x4 across all anchors. Seeds outermost so all anchors get seed=1729
# before any seed=65537 — lets you stop after one extra seed if compute
# budget tightens (still gives a 6-cell two-seed sign test).
for seed in NEW_SEEDS:
    for data_scale in ("e6", "x4"):
        for anchor in ANCHORS:
            _train_one(anchor, data_scale, seed)
print("\nseed-sweep batch training done.")


[Qwen/Qwen2.5-3B-Instruct e6 seed1729] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct e6 seed1729] run_meta.json already exists; skipping.
[google/gemma-3-4b-it e6 seed1729] run_meta.json already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct x4 seed1729] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct x4 seed1729] run_meta.json already exists; skipping.
[google/gemma-3-4b-it x4 seed1729] run_meta.json already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct e6 seed65537] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct e6 seed65537] run_meta.json already exists; skipping.
[google/gemma-3-4b-it e6 seed65537] run_meta.json already exists; skipping.
[Qwen/Qwen2.5-3B-Instruct x4 seed65537] run_meta.json already exists; skipping.
[meta-llama/Llama-3.2-3B-Instruct x4 seed65537] run_meta.json already exists; skipping.
[google/gemma-3-4b-it x4 seed65537] run_meta.json already exists; skipping.

seed-sweep batch training don

## Llama rank-seed sweep: train Llama at r in {64, 128} x seed in {1729, 65537}

Adds 4 new training runs to convert Llama's row of Table 7 (rank-sweep)
from single-seed (only seed=17) to three-seed mean +/- SE at r=64 and
r=128. Closes the reviewer concern that the v1.1 rank-ablation result
on the contrary anchor (Llama) rests on three single-seed cells.

Hyperparameters identical to the seed=17 rank-sweep baselines
(`rd-dpo-k4-bal-e6-x4-r{64,128}__seed17`); only the seed varies.
The cell is idempotent: each (rank, seed) pair is skipped if its
`run_meta.json` already exists. Estimated cost: ~3-4 A100-hours.

Source mirror at `experiments/_llama_rank_seed_sweep_train_cell.py`.


In [15]:
"""
Llama rank x seed sweep training cell.

Trains 4 new adapters: Llama-3.2-3B x {r=64, r=128} x {seed=1729, seed=65537}.
With the existing seed=17 rank-sweep adapters
(rd-dpo-k4-bal-e6-x4-r{64,128}__seed17), this gives a 3-seed average per
rank for the Llama row of Table 7 (rank-sweep), addressing the reviewer
concern that Llama's r=128 cell is the only single-seed cell in the
load-bearing rank ablation.

Mirrors the convention of the rank-sweep + seed-sweep cells:
  - same hyperparameters as the seed=17 baselines
  - per-anchor best-LR (Llama: 2e-5)
  - LoRA alpha = 2 * rank (matches rank-sweep convention)
  - rebalance with seed-keyed RNG (different seeds see different subsamples)
  - idempotent: skips runs whose run_meta.json already exists

Adapter naming: llama-3.2-3b__rd-dpo-k4-bal-e6-x4-r{rank}__seed{1729,65537}

Estimated cost: ~3-4 A100-hours total.
"""

# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# --- Llama rank x seed sweep config ---
ANCHOR    = "meta-llama/Llama-3.2-3B-Instruct"
RANKS     = [64, 128]
NEW_SEEDS = [1729, 65537]
LR        = 2e-5  # best-LR for Llama (matches Table 7 seed=17 cells)

LRS_BASE_COND = "rd-dpo-k4-bal-e6-x4"

# Hyperparameters: identical to the rank-sweep seed=17 baselines.
LRS_BETA            = 0.1
LRS_EPOCHS          = 6
LRS_WARMUP          = 2
LRS_BATCH           = 4
LRS_GA              = 8
LRS_MAX_SEQ         = 1024
LRS_LORA_DROPOUT    = 0.05
LRS_LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref", "overref_ext"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(lora_r, seed):
    short_a   = short_of(ANCHOR)
    family_a  = family_of(ANCHOR)
    cond_tag  = f"{LRS_BASE_COND}-r{lora_r}"
    run_tag   = f"{short_a}__{cond_tag}__seed{seed}"
    run_dir   = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[Llama r={lora_r} seed{seed}] run_meta.json already exists; skipping.")
        return

    print(f"\n=== Llama rank-seed-sweep training r={lora_r} seed={seed}")
    print(f"    -> {run_tag}")

    # Probe layers (top-of-net selection; same as all other Llama runs)
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"{sel_path} missing; run nb01 probe first")
    selected_blocks = json.loads(sel_path.read_text())["4"]

    # Load x4 preferences
    pairs_path = PREFS_DIR / short_a / "preferences_x4.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run nb02b x4 first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, seed)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")
    print(f"  source distribution: {dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=seed)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(LRS_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    # alpha = 2 * rank to keep effective magnitude comparable with the
    # r=16 / alpha=32 baselines (matches the rank-sweep convention).
    lora_alpha = 2 * lora_r
    lora_cfg = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, lora_dropout=LRS_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(seed)
    model_a = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=LRS_EPOCHS,
        per_device_train_batch_size=LRS_BATCH,
        gradient_accumulation_steps=LRS_GA,
        learning_rate=LR, lr_scheduler_type="cosine",
        warmup_steps=LRS_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=seed, report_to=["none"],
        beta=LRS_BETA, loss_type="sigmoid",
        max_length=LRS_MAX_SEQ,
        precompute_ref_log_probs=False,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": ANCHOR, "condition": cond_tag,
        "seed": seed, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": LRS_BETA, "lr": LR, "epochs": LRS_EPOCHS,
            "warmup_steps": LRS_WARMUP,
            "lora_r": lora_r, "lora_alpha": lora_alpha,
            "max_seq_len": LRS_MAX_SEQ,
            "per_device_train_batch_size": LRS_BATCH,
            "gradient_accumulation_steps": LRS_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


# Loop: ranks outermost so all ranks finish at seed=1729 before any seed=65537,
# letting you stop after seed=1729 (2 runs) for a 2-seed mean if budget tightens.
for seed in NEW_SEEDS:
    for lora_r in RANKS:
        _train_one(lora_r, seed)
print("\nLlama rank-seed-sweep training done.")


[Llama r=64 seed1729] run_meta.json already exists; skipping.
[Llama r=128 seed1729] run_meta.json already exists; skipping.
[Llama r=64 seed65537] run_meta.json already exists; skipping.
[Llama r=128 seed65537] run_meta.json already exists; skipping.

Llama rank-seed-sweep training done.


## Qwen+Gemma rank-seed sweep: train {Qwen, Gemma} at r in {64, 128} x seed in {1729, 65537}

Adds 8 new training runs to convert the Qwen and Gemma rows of Table 7
(rank-sweep) from single-seed (only seed=17) to three-seed mean +/- SE
at r=64 and r=128. Closes the reviewer concern that the only positive
cross-lingual cells in the rank ablation (Qwen r=128 +9.3 pp, Gemma
r=128 +1.2 pp) rest on a single seed.

Per-anchor best-LR (Qwen 2e-5, Gemma 5e-6); all other hyperparameters
identical to the seed=17 rank-sweep baselines. r=16 is NOT retrained --
the x4 r=16 cell is the seed sweep's `rd-dpo-k4-bal-e6-x4` condition,
already three-seed for both anchors. The cell is idempotent: each
(anchor, rank, seed) is skipped if its `run_meta.json` already exists.
Estimated cost: ~5-6 A100-hours.

Source mirror at `experiments/_qwen_gemma_rank_seed_sweep_train_cell.py`.


In [16]:
"""
Qwen + Gemma rank x seed sweep training cell.

Trains 8 new adapters: {Qwen-2.5-3B, Gemma-3-4B} x {r=64, r=128} x
{seed=1729, seed=65537}. With the existing seed=17 rank-sweep adapters
(rd-dpo-k4-bal-e6-x4-r{64,128}__seed17) this gives a 3-seed average per
rank for the Qwen and Gemma rows of Table 7 (rank-sweep).

Motivation (reviewer hHpd W2 / Xoi2 W2): in the v1.1 rank sweep the only
positive cross-lingual cells -- Qwen r=128 (+9.3 pp) and Gemma r=128
(+1.2 pp) -- are single-seed (seed=17), while the contrary Llama result is
already three-seed. This cell removes that asymmetry so the capacity claim
rests on three-seed mean +/- SE for every anchor. (Note: Gemma's +1.2 pp is
a single-prompt move on n=86 and is expected to be within noise; running the
seeds lets us say so with evidence rather than assertion.)

Mirrors experiments/_llama_rank_seed_sweep_train_cell.py exactly, except:
  - two anchors (Qwen, Gemma) with per-anchor best-LR (Qwen 2e-5, Gemma 5e-6);
  - r=16 is NOT retrained here -- the x4 r=16 cell is the seed-sweep's
    `rd-dpo-k4-bal-e6-x4` condition, already three-seed for both anchors.

Adapter naming: {short}__rd-dpo-k4-bal-e6-x4-r{rank}__seed{1729,65537}

Estimated cost: ~5-6 A100-hours total (Qwen r64 ~30m / r128 ~45m;
Gemma r64 ~35m / r128 ~55m; x 2 seeds).
"""

# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset, disable_caching
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# Keep the tiny in-memory preference datasets in RAM. On Colab the default
# on-disk datasets cache lands in a volatile /tmp dir that can be cleaned
# between write and read during DPOTrainer's precompute_ref_log_probs step,
# raising FileNotFoundError on cache-*.arrow. Disabling caching avoids that.
disable_caching()

# --- Qwen + Gemma rank x seed sweep config ---
# Per-anchor best-LR (matches Table 7 / x4-comparison seed=17 cells).
ANCHOR_LR = {
    "Qwen/Qwen2.5-3B-Instruct": 2e-5,
    "google/gemma-3-4b-it":     5e-6,
}
RANKS     = [64, 128]
NEW_SEEDS = [1729, 65537]

LRS_BASE_COND = "rd-dpo-k4-bal-e6-x4"

# Hyperparameters: identical to the rank-sweep seed=17 baselines.
LRS_BETA            = 0.1
LRS_EPOCHS          = 6
LRS_WARMUP          = 2
LRS_BATCH           = 4
LRS_GA              = 8
LRS_MAX_SEQ         = 1024
LRS_LORA_DROPOUT    = 0.05
LRS_LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref", "overref_ext"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor, lora_r, seed):
    lr        = ANCHOR_LR[anchor]
    short_a   = short_of(anchor)
    family_a  = family_of(anchor)
    cond_tag  = f"{LRS_BASE_COND}-r{lora_r}"
    run_tag   = f"{short_a}__{cond_tag}__seed{seed}"
    run_dir   = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{short_a} r={lora_r} seed{seed}] run_meta.json already exists; skipping.")
        return

    print(f"\n=== {short_a} rank-seed-sweep training r={lora_r} seed={seed} lr={lr:g}")
    print(f"    -> {run_tag}")

    # Probe layers (top-of-net selection; same as all other runs for this anchor)
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"{sel_path} missing; run nb01 probe first")
    selected_blocks = json.loads(sel_path.read_text())["4"]

    # Load x4 preferences
    pairs_path = PREFS_DIR / short_a / "preferences_x4.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run nb02b x4 first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, seed)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")
    print(f"  source distribution: {dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=seed)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(LRS_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    # alpha = 2 * rank to keep effective magnitude comparable with the
    # r=16 / alpha=32 baselines (matches the rank-sweep convention).
    lora_alpha = 2 * lora_r
    lora_cfg = LoraConfig(
        r=lora_r, lora_alpha=lora_alpha, lora_dropout=LRS_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(seed)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=LRS_EPOCHS,
        per_device_train_batch_size=LRS_BATCH,
        gradient_accumulation_steps=LRS_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=LRS_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=seed, report_to=["none"],
        beta=LRS_BETA, loss_type="sigmoid",
        max_length=LRS_MAX_SEQ,
        precompute_ref_log_probs=False,  # LoRA ref = adapter-disabled base; on-the-fly
        max_steps=-1,                     # avoids the datasets arrow-cache map (Colab /tmp bug)
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": cond_tag,
        "seed": seed, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": LRS_BETA, "lr": lr, "epochs": LRS_EPOCHS,
            "warmup_steps": LRS_WARMUP,
            "lora_r": lora_r, "lora_alpha": lora_alpha,
            "max_seq_len": LRS_MAX_SEQ,
            "per_device_train_batch_size": LRS_BATCH,
            "gradient_accumulation_steps": LRS_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


# Loop: seeds outermost so both anchors x both ranks finish at seed=1729
# before any seed=65537, letting you stop after seed=1729 (4 runs) for a
# 2-seed mean if compute tightens.
for seed in NEW_SEEDS:
    for anchor in ANCHOR_LR:
        for lora_r in RANKS:
            _train_one(anchor, lora_r, seed)
print("\nQwen + Gemma rank-seed-sweep training done.")


[qwen2.5-3b r=64 seed1729] run_meta.json already exists; skipping.
[qwen2.5-3b r=128 seed1729] run_meta.json already exists; skipping.
[gemma-3-4b r=64 seed1729] run_meta.json already exists; skipping.
[gemma-3-4b r=128 seed1729] run_meta.json already exists; skipping.
[qwen2.5-3b r=64 seed65537] run_meta.json already exists; skipping.
[qwen2.5-3b r=128 seed65537] run_meta.json already exists; skipping.
[gemma-3-4b r=64 seed65537] run_meta.json already exists; skipping.
[gemma-3-4b r=128 seed65537] run_meta.json already exists; skipping.

Qwen + Gemma rank-seed-sweep training done.


## Full-DPO / all-layers-LoRA baseline: train (Xoi2 suggestion 3)

Upper-bound baselines at the x4 scale. all-layers LoRA (r=16 on every
layer, matched to RD-DPO except for the probe-layer restriction) runs
across the three pre-registered seeds {17, 1729, 65537} on all three
anchors; full-model DPO (seed 17, Qwen + Llama) is the heavier literal
upper bound. Answers whether the flat cross-lingual result is caused by
restricting training to the 4 probe layers. Idempotent per (anchor,
mode, seed). Source: experiments/_full_dpo_baseline_train_cell.py.


In [17]:
"""
Full-DPO / all-layers-LoRA-DPO baseline training cell (Reviewer Xoi2 sugg. 3).

Xoi2 asked for "full DPO over all layers ... as an upper bound ... to validate
whether the conclusions from the probe-guided setting are robust." We add two
upper-bound baselines at the x4 data scale, so we can separate two hypotheses:

  1. dpo-lora-all : LoRA (r=16, alpha=32) on ALL layers, identical to RD-DPO-k4
     except the probe-layer restriction is removed. This is the matched control
     (only layer coverage differs), and it directly answers "does restricting
     to 4 probe-selected layers cause the failure?" Run across the three
     pre-registered seeds {17, 1729, 65537} so it matches the rigor of the
     rank sweep and cannot be dismissed as single-seed. Per-anchor LR as
     RD-DPO (Qwen/Llama 2e-5, Gemma 5e-6).

  2. dpo-full : full-model DPO (no LoRA), the literal upper bound. Heavier
     (~34-40 GB, A100-80G) and a smaller LR (5e-7, Zephyr-style). Seed 17 only,
     Qwen + Llama, as a supplementary point; extend to more seeds only if it
     turns out to move cross-lingual.

Reading:
  - If the all-layers / full baselines also leave held-out cross-lingual flat,
    the gap is not an artifact of RD-DPO's restricted subspace (supports the
    paper's revised, uniform-negative claim).
  - If either lifts cross-lingual, the probe-layer restriction WAS the issue,
    which contradicts the current framing and is a revision-level finding, not
    a rebuttal point.

Mirrors the rank-seed sweep cells (paths, naming, run_meta, idempotency,
rebalance, precompute).

Adapter/run naming: {short}__dpo-lora-all-bal-e6-x4__seed{S} and
{short}__dpo-full-bal-e6-x4__seed{S}.

Estimated cost: all-layers LoRA ~30-45 min/run x 9 runs (3 anchors x 3 seeds),
fits A100-40G; full-model DPO ~1.5-2.5 h/run x 2 runs, A100-80G recommended.
"""

# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, os, random, shutil, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset, enable_caching, load_from_disk
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# Full-model DPO uses precompute_ref_log_probs=True, which runs a datasets.map().
# On Colab, for an in-memory dataset that map caches to a /tmp dir that can be
# cleaned between write and read (FileNotFoundError on cache-*.arrow). Fix: keep
# caching ENABLED (a global disable_caching elsewhere would force temp files
# again) and make the splits file-backed via save_to_disk + load_from_disk, so
# map() writes its cache next to them in stable storage.
enable_caching()
_DS_CACHE_BASE = ("/content/p3_ds_cache" if os.path.isdir("/content")
                  else str(Path.home() / ".p3_ds_cache"))

# --- baseline config ---
DATA_COND  = "bal-e6-x4"                 # rebalanced, 6 epochs, x4 data
RDPO_LR    = {                            # matched to RD-DPO per-anchor best-LR
    "Qwen/Qwen2.5-3B-Instruct": 2e-5,
    "meta-llama/Llama-3.2-3B-Instruct": 2e-5,
    "google/gemma-3-4b-it": 5e-6,
}
FULL_FT_LR = 5e-7                         # full-model DPO LR (Zephyr-style)

# All-layers LoRA: matched control, three seeds, all three anchors.
LORA_ALL_ANCHORS = ["Qwen/Qwen2.5-3B-Instruct",
                    "meta-llama/Llama-3.2-3B-Instruct",
                    "google/gemma-3-4b-it"]
LORA_ALL_SEEDS   = [17, 1729, 65537]
# Full-model DPO: heavier upper bound, seed 17 only, Qwen + Llama.
FULL_ANCHORS     = ["Qwen/Qwen2.5-3B-Instruct",
                    "meta-llama/Llama-3.2-3B-Instruct"]  # Gemma full-FT: OOM-risk on 40G
FULL_SEEDS       = [17]

BETA          = 0.1
EPOCHS        = 6
WARMUP        = 2
BATCH         = 4
GA            = 8
FULL_BATCH    = 1                          # full-FT: smaller batch for memory
FULL_GA       = 32                         # keep effective batch 32
MAX_SEQ       = 1024
LORA_DROPOUT  = 0.05
LORA_R        = 16
LORA_ALPHA    = 32
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref", "overref_ext"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor, mode, seed):
    """mode in {'lora_all', 'full'}."""
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    if mode == "lora_all":
        cond_tag = f"dpo-lora-all-{DATA_COND}"
        lr = RDPO_LR[anchor]
        batch, ga = BATCH, GA
    elif mode == "full":
        cond_tag = f"dpo-full-{DATA_COND}"
        lr = FULL_FT_LR
        batch, ga = FULL_BATCH, FULL_GA
    else:
        raise ValueError(mode)

    run_tag = f"{short_a}__{cond_tag}__seed{seed}"
    run_dir = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{short_a} {mode} seed{seed}] run_meta.json already exists; skipping.")
        return

    print(f"\n=== {short_a} baseline mode={mode} seed={seed} lr={lr:g} -> {run_tag}")

    pairs_path = PREFS_DIR / short_a / "preferences_x4.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run nb02b x4 first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, seed)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full  = Dataset.from_list([_format(r) for r in pairs])
    ds_split = ds_full.train_test_split(test_size=0.05, seed=seed)
    # File-back the splits in stable storage so DPOTrainer's precompute map
    # caches next to them, not in a /tmp temp dir Colab can clean mid-run.
    _ds_dir = os.path.join(_DS_CACHE_BASE, run_tag)
    shutil.rmtree(_ds_dir, ignore_errors=True)
    ds_split.save_to_disk(_ds_dir)
    ds_split = load_from_disk(_ds_dir)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    # LoRA over ALL layers (no probe-layer regex) for lora_all; None for full.
    if mode == "lora_all":
        lora_cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS,   # list of names -> matches ALL layers
            bias="none", task_type="CAUSAL_LM",
        )
    else:
        lora_cfg = None

    set_seed(seed)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=EPOCHS,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=ga,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=seed, report_to=["none"],
        beta=BETA, loss_type="sigmoid",
        max_length=MAX_SEQ,
        # full-FT: precompute frees the reference model so full DPO fits in memory.
        # LoRA (lora_all): ref = adapter-disabled base, computed on the fly -> skips the
        # datasets.map() that writes the /tmp arrow cache Colab can drop mid-run.
        precompute_ref_log_probs=(mode == "full"),
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg,
        peft_config=lora_cfg,            # None => full-model DPO
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print(f"  starting training (mode={mode}, seed={seed})...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": cond_tag,
        "baseline_mode": mode, "seed": seed,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs), "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": BETA, "lr": lr, "epochs": EPOCHS, "warmup_steps": WARMUP,
            "lora": (None if mode == "full"
                     else {"r": LORA_R, "alpha": LORA_ALPHA,
                           "target_blocks": "all", "target_modules": LORA_TARGETS}),
            "max_seq_len": MAX_SEQ,
            "per_device_train_batch_size": batch,
            "gradient_accumulation_steps": ga,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


# All-layers LoRA-DPO first (cheap matched control, three seeds), seeds
# outermost so you can stop after seed 17 for a single-seed read if needed.
for seed in LORA_ALL_SEEDS:
    for anchor in LORA_ALL_ANCHORS:
        _train_one(anchor, "lora_all", seed)
# Full-model DPO (heavier upper bound, seed 17 only).
for seed in FULL_SEEDS:
    for anchor in FULL_ANCHORS:
        _train_one(anchor, "full", seed)
print("\nFull-DPO / all-layers-LoRA baseline training done.")


[qwen2.5-3b lora_all seed17] run_meta.json already exists; skipping.
[llama-3.2-3b lora_all seed17] run_meta.json already exists; skipping.
[gemma-3-4b lora_all seed17] run_meta.json already exists; skipping.
[qwen2.5-3b lora_all seed1729] run_meta.json already exists; skipping.
[llama-3.2-3b lora_all seed1729] run_meta.json already exists; skipping.
[gemma-3-4b lora_all seed1729] run_meta.json already exists; skipping.
[qwen2.5-3b lora_all seed65537] run_meta.json already exists; skipping.
[llama-3.2-3b lora_all seed65537] run_meta.json already exists; skipping.
[gemma-3-4b lora_all seed65537] run_meta.json already exists; skipping.

=== qwen2.5-3b baseline mode=full seed=17 lr=5e-07 -> qwen2.5-3b__dpo-full-bal-e6-x4__seed17
  loaded 331 -> rebalanced 198 pairs (99+99)


Saving the dataset (0/1 shards):   0%|          | 0/188 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

  train=188  eval=10


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  done in 5.9 min  peak=34.42 GB


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_1952/1746869992.py:241: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


  saved -> /content/drive/MyDrive/PhD/paper3-alignment/adapters/llama-3.2-3b__dpo-full-bal-e6-x4__seed17

Full-DPO / all-layers-LoRA baseline training done.
